In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=b87e94d65ae42df728cbf77bbc085f9ca9ab9c5738fca36b22e3017632b102d6
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [4]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [19]:
# ============================================================
# BB84 — Hardcoded from table (no attacker)
# ============================================================

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

_backend = AerSimulator()

# ── Data read directly from the table ────────────────────────
#    s=0 (rectilinear ⊕), d=1 (diagonal ⊗)

alice_bits  = [0,1,1,0,1,0,0,1,1,0,1,0,0,0,1,0,1,1,1,0]
alice_bases = [0,1,0,0,1,0,1,1,0,1,1,1,0,0,1,0,0,1,0,1]  # s=0, d=1
bob_bases   = [1,0,1,1,0,1,0,0,0,0,1,1,0,1,1,0,0,0,1,1]  # s=0, d=1

N = 20

# ── Encoding (Alice) ─────────────────────────────────────────

def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

# ── Measuring (Bob) ──────────────────────────────────────────

def measure_qubit(qc, basis):
    qc2 = qc.copy()
    if basis == 1:
        qc2.h(0)
    qc2.measure(0, 0)
    compiled = transpile(qc2, _backend)
    counts = _backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0])

# ── Encode all qubits ─────────────────────────────────────────

channel = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

# ── Bob measures ──────────────────────────────────────────────

bob_results = [measure_qubit(channel[i], bob_bases[i]) for i in range(N)]

# ── Display results matching table format ─────────────────────

basis_label = lambda b: 's' if b == 0 else 'd'
qubit_label = lambda bit, basis: (str(bit) if basis == 0 else ('+' if bit == 0 else '-'))

print(f"{'Index:':<10}", ''.join(f"{i:<4}" for i in range(N)))
print(f"{'A bit:':<10}", ''.join(f"{alice_bits[i]:<4}"              for i in range(N)))
print(f"{'A basis:':<10}", ''.join(f"{basis_label(alice_bases[i]):<4}" for i in range(N)))
print(f"{'qubit:':<10}", ''.join(f"{qubit_label(alice_bits[i], alice_bases[i]):<4}" for i in range(N)))
print(f"{'B basis:':<10}", ''.join(f"{basis_label(bob_bases[i]):<4}" for i in range(N)))

# Bob's bit: show result only where bases match, else '?'
bob_display = [
    str(bob_results[i]) if alice_bases[i] == bob_bases[i] else '?'
    for i in range(N)
]
print(f"{'B bit:':<10}", ''.join(f"{bob_display[i]:<4}" for i in range(N)))

# ── Sifting ───────────────────────────────────────────────────

matching  = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i]  for i in matching]
bob_key   = [bob_results[i] for i in matching]

print(f"\nMatching positions : {matching}")
print(f"Alice's sifted key : {alice_key}")
print(f"Bob's   sifted key : {bob_key}")

errors     = sum(a != b for a, b in zip(alice_key, bob_key))
error_rate = errors / len(alice_key) if alice_key else 0
print(f"\nErrors: {errors}/{len(alice_key)} = {error_rate:.1%}")
print("✓ Keys match!" if alice_key == bob_key else "✗ Keys differ!")

Index:     0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  
A bit:     0   1   1   0   1   0   0   1   1   0   1   0   0   0   1   0   1   1   1   0   
A basis:   s   d   s   s   d   s   d   d   s   d   d   d   s   s   d   s   s   d   s   d   
qubit:     0   -   1   0   -   0   +   -   1   +   -   +   0   0   -   0   1   -   1   +   
B basis:   d   s   d   d   s   d   s   s   s   s   d   d   s   d   d   s   s   s   d   d   
B bit:     ?   ?   ?   ?   ?   ?   ?   ?   1   ?   1   0   0   ?   1   0   1   ?   ?   0   

Matching positions : [8, 10, 11, 12, 14, 15, 16, 19]
Alice's sifted key : [1, 1, 0, 0, 1, 0, 1, 0]
Bob's   sifted key : [1, 1, 0, 0, 1, 0, 1, 0]

Errors: 0/8 = 0.0%
✓ Keys match!
